In [1]:
import inspect
import os
import sys

from flax import nnx
from flax.typing import Dtype, Shape
from typing import Callable, Any, Optional

import jax
import jax.numpy as jnp
import numpy as np


"Referenced from https://github.com/huggingface/transformers/blob/v4.57.0/src/transformers/models/roformer/modeling_roformer.py"
def apply_rotary_position_embeddings(
    sinusoidal_pos_embed: jax.Array,
    queries: jax.Array,
    keys: jax.Array,
    num_heads: int,
    embed_dim_per_head: int,
):
    # MultiHeadAttention treats q, k, v as (batch, seq_len, num_heads, head_dim)
    sin, cos = jnp.split(sinusoidal_pos_embed, 2, axis=-1)
    bs, seq_len, _ = queries.shape

    sin = sin.reshape(
        (bs, seq_len, 1, embed_dim_per_head // 2)
    )
    cos = cos.reshape(
        (bs, seq_len, 1, embed_dim_per_head // 2)
    )

    sin_pos = jnp.stack([sin, sin], axis=-1).reshape(
        (bs, seq_len, 1, embed_dim_per_head)
    )
    cos_pos = jnp.stack([cos, cos], axis=-1).reshape(
        (bs, seq_len, 1, embed_dim_per_head)
    )

    queries = queries.reshape(
        (bs, seq_len, num_heads, embed_dim_per_head)
    )
    keys = keys.reshape(
        (bs, seq_len, num_heads, embed_dim_per_head)
    )

    rotate_half_queries = jnp.stack(
        (-queries[..., 1::2], queries[..., ::2]),
        axis=-1,
    ).reshape(queries.shape)
    queries = queries * cos_pos + rotate_half_queries * sin_pos

    rotate_half_keys = jnp.stack(
        (-keys[..., 1::2], keys[..., ::2]),
        axis=-1,
    ).reshape(keys.shape)
    keys = keys * cos_pos + rotate_half_keys * sin_pos

    queries = queries.reshape((bs, seq_len, -1))
    keys = keys.reshape((bs, seq_len, -1))

    return queries, keys

In [2]:
def get_sinusoidal_position_embedding(
    positions: jax.Array,
    embed_dim_per_head: int,
    n_value: float = 10000.0,
) -> jax.Array:
    # positions: [batch, sequence_length,]
    thetas = positions.flatten()[:, None] / jnp.power(
        n_value, 2 * (jnp.arange(embed_dim_per_head) // 2) / embed_dim_per_head
    )[None]
    thetas = thetas.reshape((*positions.shape, embed_dim_per_head))

    out_sin = jnp.sin(thetas[..., 0::2])
    out_cos = jnp.cos(thetas[..., 1::2])
    return jnp.concatenate((out_sin, out_cos), axis=-1)

In [21]:
embeddings = np.arange(640).reshape(10, 64) / 640

In [22]:
seq_1 = np.stack([embeddings[0], embeddings[0], embeddings[1], embeddings[2]])[None]
seq_2 = np.stack([embeddings[0], embeddings[1], embeddings[2], embeddings[0]])[None]

In [23]:
positions = jnp.tile(
    jnp.arange(seq_1.shape[1]),
    reps=(seq_1.shape[0], 1),
)

In [24]:
sinusoidal_pos_embed = get_sinusoidal_position_embedding(
    positions,
    8,
    10000.0,
)

In [25]:
sinusoidal_pos_embed

Array([[[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
          1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00],
        [ 8.4147102e-01,  9.9833421e-02,  9.9998331e-03,  9.9999993e-04,
          5.4030228e-01,  9.9500418e-01,  9.9994999e-01,  9.9999952e-01],
        [ 9.0929747e-01,  1.9866933e-01,  1.9998666e-02,  1.9999987e-03,
         -4.1614681e-01,  9.8006660e-01,  9.9980003e-01,  9.9999797e-01],
        [ 1.4112000e-01,  2.9552022e-01,  2.9995499e-02,  2.9999956e-03,
         -9.8999250e-01,  9.5533651e-01,  9.9955004e-01,  9.9999553e-01]]],      dtype=float32)

In [26]:
res_1 = apply_rotary_position_embeddings(
    sinusoidal_pos_embed=sinusoidal_pos_embed,
    queries=seq_1,
    keys=seq_1,
    num_heads=8,
    embed_dim_per_head=8,
)

res_2 = apply_rotary_position_embeddings(
    sinusoidal_pos_embed=sinusoidal_pos_embed,
    queries=seq_2,
    keys=seq_2,
    num_heads=8,
    embed_dim_per_head=8,
)

In [29]:
res_1[0][0, ..., :8] @ res_1[1][0, ..., :8].T

Array([[0.0003418 , 0.00034051, 0.00465211, 0.00853929],
       [0.00034051, 0.0003418 , 0.00478851, 0.00896654],
       [0.00465211, 0.00478851, 0.0890918 , 0.15479714],
       [0.00853929, 0.00896654, 0.15479714, 0.3378418 ]], dtype=float32)

In [30]:
res_2[0][0, ..., :8] @ res_2[1][0, ..., :8].T

Array([[3.4179687e-04, 4.7885063e-03, 8.9665391e-03, 3.3547499e-04],
       [4.7885063e-03, 8.9091800e-02, 1.5479714e-01, 4.2989980e-03],
       [8.9665391e-03, 1.5479714e-01, 3.3784181e-01, 8.6413119e-03],
       [3.3547499e-04, 4.2989980e-03, 8.6413119e-03, 3.4179693e-04]],      dtype=float32)